In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("env", "dev", "Ambiente (dev | prod)")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog_base", "catalog_pr_db")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsdvbtdev01")


env = dbutils.widgets.get("env").strip().lower()
container = dbutils.widgets.get("container")
catalogo = f"{dbutils.widgets.get('catalog_base')}_{env}"
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
 
assert env in ("dev", "prod"), f"env inválido: {env}"
 
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/constructors.csv"
print(f"env={env} | catalogo={catalogo} | ruta={ruta}")


In [0]:
df_constructors = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
df_constructors.display()

In [0]:
constructors_schema = StructType([
    StructField("constructorId", IntegerType(), False),
    StructField("constructorRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
constructors_schema

In [0]:
df_constructs_final = spark.read\
.option('header', True)\
.schema(constructors_schema)\
.csv(ruta)

In [0]:
constructs_selected_df = df_constructs_final.select(
    col("constructorId"),
    col("constructorRef"),
    col("name"),
    col("nationality")
)

In [0]:
constructors_renamed_df = constructs_selected_df.withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("constructorRef", "constructor_ref")
   

In [0]:

constructors_final_df = constructors_renamed_df \
    .withColumn("ingestion_date", current_timestamp())

constructors_final_df.display()

In [0]:
print(f"catalogo = {catalogo}")
print(f"esquema = {esquema}")

In [0]:
spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"USE SCHEMA `{esquema}`")

In [0]:
print(f"current_catalog = {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"current_schema = {spark.sql('SELECT current_schema()').collect()[0][0]}")


In [0]:
constructors_final_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.constructors")

In [0]:
%sql
--select * from catalog_pr_db_dev.bronze.constructors